# 📈 Análisis Financiero de Mercados
### Portfolio Project · Python + Pandas + Plotly

**Objetivo:** Analizar el rendimiento histórico de activos financieros, identificar tendencias y visualizar métricas clave de riesgo/retorno.

**Skills utilizados:** `pandas` · `numpy` · `plotly` · `scipy` · análisis estadístico · visualización de datos

---

## 1. 📦 Importación de Librerías

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

print('✅ Librerías cargadas correctamente')

## 2. 📊 Generación de Datos Simulados

> **Nota:** En un proyecto real, estos datos se obtendrían de APIs como `yfinance`, `Alpha Vantage` o `Yahoo Finance`. Aquí simulamos datos realistas con distribuciones estadísticas correctas.

In [ ]:
np.random.seed(42)

activos = {
    'AAPL':  {'mu': 0.0008, 'sigma': 0.018, 'precio_inicial': 150},
    'GOOGL': {'mu': 0.0006, 'sigma': 0.016, 'precio_inicial': 130},
    'MSFT':  {'mu': 0.0009, 'sigma': 0.015, 'precio_inicial': 280},
    'AMZN':  {'mu': 0.0005, 'sigma': 0.020, 'precio_inicial': 170},
    'BTC':   {'mu': 0.0015, 'sigma': 0.045, 'precio_inicial': 35000},
}

fechas = pd.date_range(start='2022-01-03', periods=504, freq='B')
precios = {}

for ticker, params in activos.items():
    retornos = np.random.normal(params['mu'], params['sigma'], 504)
    precio = params['precio_inicial'] * np.exp(np.cumsum(retornos))
    precios[ticker] = precio

df_precios = pd.DataFrame(precios, index=fechas)

print(f'✅ Dataset generado: {df_precios.shape[0]} días de trading × {df_precios.shape[1]} activos')
print(f'\n📅 Período: {fechas[0].date()} → {fechas[-1].date()}')
df_precios.tail()

## 3. 📉 Análisis de Retornos

In [ ]:
df_retornos = df_precios.pct_change().dropna()
df_retornos_acum = (1 + df_retornos).cumprod() - 1

metricas = pd.DataFrame({
    'Retorno Anual (%)':     (df_retornos.mean() * 252 * 100).round(2),
    'Volatilidad Anual (%)': (df_retornos.std() * np.sqrt(252) * 100).round(2),
    'Retorno Total (%)':     (df_retornos_acum.iloc[-1] * 100).round(2),
    'Sharpe Ratio':          ((df_retornos.mean() * 252) / (df_retornos.std() * np.sqrt(252))).round(3),
    'Max Drawdown (%)':      (df_retornos_acum.apply(lambda x: (x - x.cummax()).min()) * 100).round(2),
})

print('📊 MÉTRICAS DE RENDIMIENTO')
print('=' * 60)
metricas

## 4. 📈 Visualización: Rendimiento Acumulado

In [ ]:
colores = ['#00D4AA', '#FF6B6B', '#4ECDC4', '#FFE66D', '#F7B731']

fig = go.Figure()

for i, ticker in enumerate(df_retornos_acum.columns):
    fig.add_trace(go.Scatter(
        x=df_retornos_acum.index,
        y=df_retornos_acum[ticker] * 100,
        name=ticker,
        line=dict(color=colores[i], width=2.5),
        hovertemplate=f'<b>{ticker}</b><br>Fecha: %{{x|%d %b %Y}}<br>Retorno: %{{y:.1f}}%<extra></extra>'
    ))

fig.add_hline(y=0, line_dash='dash', line_color='rgba(255,255,255,0.3)', line_width=1)

fig.update_layout(
    title=dict(text='📈 Retorno Acumulado (2022–2024)', font=dict(size=22, color='white')),
    plot_bgcolor='#1a1a2e',
    paper_bgcolor='#16213e',
    font=dict(color='#e0e0e0'),
    xaxis=dict(gridcolor='rgba(255,255,255,0.05)', showgrid=True),
    yaxis=dict(gridcolor='rgba(255,255,255,0.05)', showgrid=True, ticksuffix='%', title='Retorno (%)'),
    legend=dict(bgcolor='rgba(0,0,0,0.4)', bordercolor='rgba(255,255,255,0.1)', borderwidth=1),
    hovermode='x unified',
    height=500,
)

fig.show()

## 5. 🎯 Mapa de Riesgo vs Retorno (Frontera Eficiente)

In [ ]:
n_portafolios = 5000
n_activos = len(df_retornos.columns)
tickers = df_retornos.columns.tolist()

port_retornos = []
port_volatilidades = []
port_sharpes = []
port_pesos = []

for _ in range(n_portafolios):
    pesos = np.random.dirichlet(np.ones(n_activos))
    ret = np.dot(pesos, df_retornos.mean()) * 252
    cov = df_retornos.cov() * 252
    vol = np.sqrt(np.dot(pesos.T, np.dot(cov, pesos)))
    sharpe = ret / vol
    port_retornos.append(ret * 100)
    port_volatilidades.append(vol * 100)
    port_sharpes.append(sharpe)
    port_pesos.append(pesos)

idx_optimo = np.argmax(port_sharpes)
pesos_optimos = dict(zip(tickers, (port_pesos[idx_optimo] * 100).round(1)))

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=port_volatilidades, y=port_retornos,
    mode='markers',
    marker=dict(
        color=port_sharpes,
        colorscale='Viridis',
        size=3,
        colorbar=dict(
            title=dict(text='Sharpe Ratio', font=dict(color='white')),
            tickfont=dict(color='white')
        ),
        opacity=0.6
    ),
    name='Portafolios',
    hovertemplate='Volatilidad: %{x:.1f}%<br>Retorno: %{y:.1f}%<extra></extra>'
))

fig.add_trace(go.Scatter(
    x=[port_volatilidades[idx_optimo]], y=[port_retornos[idx_optimo]],
    mode='markers+text',
    marker=dict(color='#FF6B6B', size=16, symbol='star', line=dict(color='white', width=2)),
    text=['⭐ Óptimo'],
    textposition='top right',
    textfont=dict(color='#FF6B6B', size=12),
    name=f'Sharpe Máximo: {max(port_sharpes):.2f}'
))

fig.update_layout(
    title=dict(text='🎯 Frontera Eficiente — Simulación Monte Carlo (5,000 portafolios)', font=dict(size=20, color='white')),
    plot_bgcolor='#1a1a2e', paper_bgcolor='#16213e',
    font=dict(color='#e0e0e0'),
    xaxis=dict(title='Volatilidad Anual (%)', gridcolor='rgba(255,255,255,0.05)', ticksuffix='%'),
    yaxis=dict(title='Retorno Anual (%)', gridcolor='rgba(255,255,255,0.05)', ticksuffix='%'),
    height=520,
)

print(f'\n⭐ PORTAFOLIO ÓPTIMO (Sharpe: {max(port_sharpes):.3f})')
print('Distribución sugerida:')
for ticker, peso in pesos_optimos.items():
    print(f'  {ticker:6s}: {peso:.1f}%')

fig.show()

## 6. 🔥 Mapa de Correlación entre Activos

In [ ]:
corr = df_retornos.corr()

fig = go.Figure(data=go.Heatmap(
    z=corr.values,
    x=corr.columns,
    y=corr.index,
    colorscale=[
        [0.0, '#FF6B6B'],
        [0.5, '#1a1a2e'],
        [1.0, '#00D4AA']
    ],
    zmin=-1, zmax=1,
    text=corr.round(2).values,
    texttemplate='%{text}',
    textfont=dict(size=14, color='white'),
    hovertemplate='%{x} / %{y}<br>Correlación: %{z:.3f}<extra></extra>'
))

fig.update_layout(
    title=dict(text='🔥 Matriz de Correlación de Retornos Diarios', font=dict(size=20, color='white')),
    plot_bgcolor='#1a1a2e', paper_bgcolor='#16213e',
    font=dict(color='#e0e0e0'),
    xaxis=dict(tickfont=dict(size=13, color='#00D4AA')),
    yaxis=dict(tickfont=dict(size=13, color='#00D4AA')),
    height=460,
)

fig.show()

## 7. 📊 Dashboard Final: Resumen Ejecutivo

In [ ]:
fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=[
        'Retorno Anual vs Volatilidad',
        'Sharpe Ratio por Activo',
        'Distribución de Retornos Diarios',
        'Precio Normalizado (Base 100)'
    ],
    vertical_spacing=0.15,
    horizontal_spacing=0.12
)

tickers_acciones = ['AAPL', 'GOOGL', 'MSFT', 'AMZN', 'BTC']

for i, ticker in enumerate(tickers_acciones):
    fig.add_trace(go.Scatter(
        x=[metricas.loc[ticker, 'Volatilidad Anual (%)']],
        y=[metricas.loc[ticker, 'Retorno Anual (%)']],
        mode='markers+text',
        marker=dict(color=colores[i], size=14),
        text=[ticker], textposition='top center',
        textfont=dict(color=colores[i], size=11),
        name=ticker, showlegend=False
    ), row=1, col=1)

fig.add_trace(go.Bar(
    x=tickers_acciones,
    y=metricas['Sharpe Ratio'].values,
    marker_color=colores,
    showlegend=False
), row=1, col=2)

for i, ticker in enumerate(tickers_acciones[:3]):
    fig.add_trace(go.Histogram(
        x=df_retornos[ticker] * 100,
        name=ticker,
        opacity=0.6,
        marker_color=colores[i],
        showlegend=False,
        nbinsx=40
    ), row=2, col=1)

df_norm = df_precios / df_precios.iloc[0] * 100
for i, ticker in enumerate(tickers_acciones):
    fig.add_trace(go.Scatter(
        x=df_norm.index, y=df_norm[ticker],
        name=ticker, line=dict(color=colores[i], width=1.8),
        showlegend=False
    ), row=2, col=2)

fig.update_layout(
    title=dict(text='📊 Dashboard Financiero — Resumen Ejecutivo', font=dict(size=22, color='white'), x=0.5),
    plot_bgcolor='#1a1a2e', paper_bgcolor='#0f0f23',
    font=dict(color='#e0e0e0'),
    height=700, barmode='overlay',
)

for ann in fig.layout.annotations:
    ann.font.color = '#00D4AA'
    ann.font.size = 13

fig.show()

print('\n✅ Análisis completo. Dashboard listo para compartir en LinkedIn.')

---
## 8. 💡 Conclusiones

| Activo | Perfil |
|--------|--------|
| **MSFT** | Mejor balance riesgo/retorno (Sharpe más alto entre acciones) |
| **BTC**  | Mayor retorno potencial pero volatilidad ~3x superior a las acciones |
| **AAPL** | Retorno sólido con volatilidad moderada |
| **GOOGL/AMZN** | Retornos más conservadores, buena diversificación |

### Portafolio Óptimo Sugerido
La simulación Monte Carlo identificó que una distribución que maximiza el Sharpe Ratio tiende a **sobreponderar activos de menor volatilidad** (MSFT, GOOGL) como ancla, con una posición moderada en BTC como componente de alto crecimiento.

> ⚠️ **Disclaimer:** Este análisis es con fines educativos y de portfolio profesional. No constituye asesoramiento financiero.